In [ ]:
import os
import numpy as np
import tempfile
import shutil
from ptm_ice.ice_depth_pipeline import run_pipeline, write_csv

## 1. Create Synthetic Binary Temperature Data

Simulate a small PTM3D run: 100 faces, 20 timesteps, varying temperatures.

In [ ]:
n_faces = 100
n_timesteps = 20

# Create temp directory
temp_dir = tempfile.mkdtemp()
print(f"Temp directory: {temp_dir}")

# Simulate temperature evolution: sinusoidal variation over time + spatial gradient
for t in range(n_timesteps):
    phase = 2 * np.pi * t / n_timesteps
    base_temp = 90.0 + 20.0 * np.sin(phase)
    
    # Spatial gradient: faces 0-49 warmer, 50-99 cooler
    temps = np.linspace(base_temp + 10, base_temp - 10, n_faces).astype('float32')
    
    # Add some noise
    temps += np.random.uniform(-2, 2, n_faces).astype('float32')
    
    # Write binary file
    filename = os.path.join(temp_dir, f'temperature_{t:04d}.bin')
    temps.tofile(filename)

print(f"Created {n_timesteps} binary temperature files")

## 2. Run the Pipeline

Process the binary files to compute ice depths. The pipeline will:
1. Stream through all timesteps
2. Compute mean and max temperature per face
3. Apply vectorized ice depth solver
4. Write CSV output

In [ ]:
out_dir = os.path.join(temp_dir, 'ice_depth_output')

csv_path = run_pipeline(
    source_dir=temp_dir,
    fmt='binary',
    timesteps=n_timesteps,
    out_dir=out_dir,
    dtype='float32',
    backend='numpy',  # Use 'auto' to enable Numba if available
    chunk_size=50,    # Small chunk for demo
    # Physical parameters (defaults shown)
    J0=1e-5,
    delta=0.05,
    l=2.83e6,
    zs=4000.0,
    Q=1e-3,
    k=205.0,
)

print(f"Pipeline complete. CSV output: {csv_path}")

## 3. Load and Inspect Results

In [ ]:
import csv

with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Loaded {len(rows)} rows")
print("\nFirst 5 entries:")
for row in rows[:5]:
    print(f"Face {row['face']}: T_mean={float(row['T_mean']):.2f} K, "
          f"T_max={float(row['T_max']):.2f} K, "
          f"ice_depth={row['ice_depth']} m")

## 4. Visualize Ice Depths

Plot ice depth vs mean temperature to see the physical relationship.

In [ ]:
import matplotlib.pyplot as plt

T_mean = np.array([float(r['T_mean']) for r in rows])
T_max = np.array([float(r['T_max']) for r in rows])
depths = np.array([float(r['ice_depth']) if r['ice_depth'] != 'nan' else np.nan for r in rows])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Ice depth vs mean temperature
mask = ~np.isnan(depths)
ax1.scatter(T_mean[mask], depths[mask], alpha=0.6, s=30)
ax1.set_xlabel('Mean Temperature (K)', fontsize=12)
ax1.set_ylabel('Ice Depth (m)', fontsize=12)
ax1.set_title('Ice Depth vs Mean Temperature', fontsize=14)
ax1.grid(alpha=0.3)

# Spatial variation
face_indices = np.arange(n_faces)
ax2.plot(face_indices, T_mean, label='T_mean', marker='o', markersize=3, alpha=0.7)
ax2.plot(face_indices, T_max, label='T_max', marker='s', markersize=3, alpha=0.7)
ax2.set_xlabel('Face Index', fontsize=12)
ax2.set_ylabel('Temperature (K)', fontsize=12)
ax2.set_title('Spatial Temperature Distribution', fontsize=14)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(f"  Valid depths: {np.sum(mask)} / {len(depths)}")
print(f"  Depth range: [{np.nanmin(depths):.6f}, {np.nanmax(depths):.6f}] m")
print(f"  Mean depth: {np.nanmean(depths):.6f} m")

## 5. Cleanup

In [ ]:
shutil.rmtree(temp_dir)
print(f"Cleaned up temp directory: {temp_dir}")

## 6. CLI Usage Example

For production runs on actual PTM3D outputs, use the command-line interface:

```bash
# Binary format
python -m ptm_ice.ice_depth_pipeline \
    --source-dir /path/to/ptm_output/temperature \
    --format binary \
    --timesteps 720 \
    --dtype float32 \
    --backend auto \
    --out-dir /path/to/ice_depth_results \
    --chunk-size 1000000 \
    --hdf5

# Legacy VTK format
python -m ptm_ice.ice_depth_pipeline \
    --source-dir /path/to/vtk_files \
    --format vtk \
    --timesteps 0 \
    --backend auto \
    --out-dir /path/to/ice_depth_results
```

**Performance Notes:**
- For ~1e7 faces, expect ~30-60 seconds on modern hardware with Numba backend
- Memory usage scales with `chunk_size` (default 500k faces ~ 20-40 MB working set)
- HDF5 output requires `h5py` package